In [2]:
import numpy as np
import pandas as pd

# 加载数据
data = pd.read_pickle('/root/flight_data.pkl')
header_df = pd.read_csv('/root/flight_header.csv')

print(f"data 中航班数: {len(data)}")
print(f"header 中航班数: {len(header_df)}")
print(f"header 列名: {header_df.columns.tolist()}")

data 中航班数: 11446
header 中航班数: 11446
header 列名: ['Master Index', 'before_after', 'date_diff', 'flight_length', 'label', 'hierarchy', 'fold', 'class', 'target_class', 'hclass', 'number_flights_before']


In [4]:
import os
import sys
import argparse

# 检测是否在 Jupyter 环境中
if 'ipykernel' in sys.modules:
    # Notebook 环境：手动修改这里
    data_dir = '/root'  # ⚠️ 改成你实际的路径
else:
    # 命令行环境：从参数读取
    parser = argparse.ArgumentParser()
    parser.add_argument('--data_dir', type=str, required=True,
                        help='Path to NGAFID data directory')
    args, _ = parser.parse_known_args()
    data_dir = args.data_dir

print(f"使用数据目录: {data_dir}")

data = pd.read_pickle(os.path.join(data_dir, 'flight_data.pkl'))
header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))
stats_df = pd.read_csv(os.path.join(data_dir, 'stats.csv'), index_col=0)

使用数据目录: /root


In [20]:
import numpy as np
import pandas as pd
import time

# ========== 1. 数据已加载（来自单元格2） ==========
print(f"data 航班数: {len(data)}, header 航班数: {len(header_df)}")

# ========== 2. 筛选事件 ==========
# 论文 Section 3.2 的 19 类维护事件基准子集
# 从 header_df 中筛选出所有维护事件（非空 label）
# 注意：你需要排除掉不相关的事件或空值，这里只取 label 列非空的样本
event_flights = header_df[header_df['label'].notna()]
event_ids = event_flights['Master Index'].values

print(f"19类基准子集（label非空）涉及的航班数: {len(event_ids)}")
print(f"涉及的航班数（原始）: {len(event_ids)}")

# 找出 data 和 header_df 中共有的 Master Index
data_keys = set(data.keys())
header_indices = set(header_df['Master Index'].values)
common_ids = list(data_keys & header_indices)

# 用共有的 ID 重新筛选（确保每个 flight_id 都能在 data 中找到数据）
event_ids_common = [idx for idx in event_ids if idx in common_ids]
print(f"筛选后的事件航班数（与 data 对齐）: {len(event_ids_common)}")

# ========== 3. 定义辅助函数 ==========
def pad_or_truncate(seq, target_len=3000):
    """将序列截断或填充到固定长度"""
    current_len = seq.shape[0]
    if current_len >= target_len:
        return seq[:target_len, :]
    else:
        pad_width = ((0, target_len - current_len), (0, 0))
        return np.pad(seq, pad_width, mode='constant', constant_values=0)

def normalize(seq, stats_df):
    """使用 stats.csv 中的 min/max 归一化"""
    for i in range(seq.shape[1]):
        col_name = f'sensor_{i}'
        if col_name in stats_df.index:
            min_val = stats_df.loc[col_name, 'min']
            max_val = stats_df.loc[col_name, 'max']
            if max_val > min_val:
                seq[:, i] = (seq[:, i] - min_val) / (max_val - min_val)
    return seq

# ========== 4. 准备数据（使用 event_ids_common） ==========
# ========== 4. 准备数据（使用 event_ids_common） ==========
target_len = 3000
X_list = []
y = []

print("准备数据...")
for flight_id in event_ids_common:
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    sensor_data = normalize(sensor_data, stats_df)
    sensor_data = pad_or_truncate(sensor_data, target_len)
    X_list.append(sensor_data)
    # 直接从 header_df 中按 'Master Index' 列查找
    label = header_df[header_df['Master Index'] == flight_id]['before_after'].values[0]
    y.append(label)

X = np.array(X_list)
y = np.array(y)

# 检查时间顺序：直接用 event_ids_common 和 header_df 的 'Master Index' 列
date_diff_series = header_df.set_index('Master Index').loc[event_ids_common, 'date_diff']
date_diff = date_diff_series.values
if np.all(np.diff(date_diff) >= 0):
    print("✅ 数据已按时间排序")
else:
    print("⚠️ 数据未按时间排序，进行重排...")
    sorted_idx = np.argsort(date_diff)
    X = X[sorted_idx]
    y = y[sorted_idx]
    print("✅ 数据已重排")

print(f"X 形状: {X.shape}")
print(f"标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

data 航班数: 11446, header 航班数: 11446
19类基准子集（label非空）涉及的航班数: 11446
涉及的航班数（原始）: 11446
筛选后的事件航班数（与 data 对齐）: 11446
准备数据...
⚠️ 数据未按时间排序，进行重排...
✅ 数据已重排
X 形状: (11446, 3000, 23)
标签分布: 0=5844, 1=5602


In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sklearn.linear_model import LogisticRegressionCV

# ========== 5. 使用 StratifiedKFold（不打乱） ==========
minirocket = MiniRocketMultivariate(random_state=42)
classifier = LogisticRegressionCV(Cs=10, cv=5, random_state=42, max_iter=1000)

# 关键修改：用 StratifiedKFold 替代 TimeSeriesSplit，保持 shuffle=False
cv = StratifiedKFold(n_splits=5, shuffle=False)
accuracies, f1_scores, auc_scores = [], [], []

print("\n开始分层交叉验证（保持顺序）...")
fold = 1
for train_idx, val_idx in cv.split(X, y):
    print(f"\n--- Fold {fold} ---")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # 检查当前折的标签分布（防止意外）
    print(f"train: 0={sum(y_train==0)}, 1={sum(y_train==1)} | val: 0={sum(y_val==0)}, 1={sum(y_val==1)}")
    
    start = time.time()
    X_train_transform = minirocket.fit_transform(X_train, y_train)
    classifier.fit(X_train_transform, y_train)
    X_val_transform = minirocket.transform(X_val)
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    train_time = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {train_time:.2f}s")
    fold += 1

print("\n" + "="*50)
print("分层交叉验证最终结果:")
print(f"准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*50)


开始分层交叉验证（保持顺序）...

--- Fold 1 ---
train: 0=4675, 1=4481 | val: 0=1169, 1=1121


/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Plea

准确率: 0.5410, F1: 0.5298, AUC: 0.5609, 耗时: 318.54s

--- Fold 2 ---
train: 0=4676, 1=4481 | val: 0=1168, 1=1121


/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Plea

准确率: 0.5443, F1: 0.5291, AUC: 0.5480, 耗时: 311.52s

--- Fold 3 ---
train: 0=4675, 1=4482 | val: 0=1169, 1=1120


...

准确率: 0.5352, F1: 0.5133, AUC: 0.5386, 耗时: 313.34s

--- Fold 4 ---
train: 0=4675, 1=4482 | val: 0=1169, 1=1120


/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Plea

准确率: 0.5491, F1: 0.5213, AUC: 0.5705, 耗时: 313.45s

--- Fold 5 ---
train: 0=4675, 1=4482 | val: 0=1169, 1=1120


/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/root/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Plea

准确率: 0.5688, F1: 0.5298, AUC: 0.5922, 耗时: 293.42s

分层交叉验证最终结果:
准确率: 0.5477 ± 0.0115
F1分数: 0.5247 ± 0.0065
AUC: 0.5620 ± 0.0186


In [2]:
import numpy as np
import pandas as pd
import os

# ==========================================
# 1. 加载数据（请确保路径正确）
# ==========================================
# 如果你在云端，data_dir 可能是 '/root' 或其他
data_dir = '/root'  # ⚠️ 请根据你的实际环境修改

data = pd.read_pickle(os.path.join(data_dir, 'flight_data.pkl'))
header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))
stats_df = pd.read_csv(os.path.join(data_dir, 'stats.csv'), index_col=0)

print(f"✅ 数据加载成功: data={len(data)} 航班, header={len(header_df)} 航班")

# ==========================================
# 2. 检查所有列名
# ==========================================
print("\n📋 header_df 全部列名:")
print(header_df.columns.tolist())

# ==========================================
# 3. 检查关键标签列的内容（前20行）
# ==========================================
print("\n📊 关键列前20行样本:")
print(header_df[['Master Index', 'before_after', 'label', 'target_class', 'class', 'date_diff']].head(20))

# ==========================================
# 4. 统计各标签列分布
# ==========================================
print("\n📈 before_after 分布:")
print(header_df['before_after'].value_counts().sort_index())

print("\n📈 label 分布（前20个）:")
print(header_df['label'].value_counts().head(20))

print("\n📈 target_class 分布（前20个）:")
print(header_df['target_class'].value_counts().head(20))

# ==========================================
# 5. 检查是否有 NaN 及数据类型
# ==========================================
print("\n🔍 关键列缺失值统计:")
print(header_df[['before_after', 'label', 'target_class', 'class']].isnull().sum())

print("\n🔍 关键列数据类型:")
print(header_df[['before_after', 'label', 'target_class', 'class']].dtypes)

# ==========================================
# 6. 分析 before_after 与 label 的关系（交叉表）
# ==========================================
print("\n🔗 before_after vs label 交叉表（前10类）:")
cross_tab = pd.crosstab(header_df['before_after'], header_df['label'])
print(cross_tab.iloc[:, :10])  # 只显示前10列，避免太长

# ==========================================
# 7. 依据论文 Section 3.2 标准，检查候选子集
# ==========================================
print("\n📐 论文 Section 3.2 子集标准：")
print("  - date_diff ∈ [-2, -1] (维护前2天)")
print("  - date_diff ∈ [1, 2]  (维护后2天)")
print("  - 排除 date_diff == 0 (维护当天)")
print("  - 排除 label 为空的样本")
print("  - 只保留每个 label 类别至少有50个样本的类别")

# 先看原始数据中，date_diff 在 [-2, -1] 和 [1, 2] 范围内的样本数
mask_candidate = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
candidate_df = header_df[mask_candidate]
print(f"\n候选样本数（date_diff在±2天内且非0）: {len(candidate_df)}")

# 再看这些候选样本的 before_after 分布
print(f"\n候选样本中 before_after 分布:")
print(candidate_df['before_after'].value_counts().sort_index())

# 检查候选样本中 label 的类别数
print(f"\n候选样本中 label 的唯一值数: {candidate_df['label'].nunique()}")

# 统计每个 label 类别在候选样本中的数量
label_counts = candidate_df['label'].value_counts()
print(f"\n每个 label 类别的样本数（前20个）:")
print(label_counts.head(20))

# 筛选出样本数 >= 50 的类别
valid_labels = label_counts[label_counts >= 50].index.tolist()
print(f"\n样本数 >= 50 的类别数: {len(valid_labels)}")
print(f"这些类别是: {sorted(valid_labels)}")

# 最终筛选
final_mask = mask_candidate & (header_df['label'].isin(valid_labels))
final_df = header_df[final_mask]
print(f"\n✅ 论文 Section 3.2 标准筛选后的样本数: {len(final_df)}")
print(f"  其中 before_after=0 (维护后): {sum(final_df['before_after']==0)}")
print(f"  其中 before_after=1 (维护前): {sum(final_df['before_after']==1)}")

✅ 数据加载成功: data=11446 航班, header=11446 航班

📋 header_df 全部列名:
['Master Index', 'before_after', 'date_diff', 'flight_length', 'label', 'hierarchy', 'fold', 'class', 'target_class', 'hclass', 'number_flights_before']

📊 关键列前20行样本:
    Master Index  before_after                      label  target_class  \
0              1             1  intake gasket leak/damage            10   
1              2             1  intake gasket leak/damage            10   
2              7             0  intake gasket leak/damage             0   
3              9             1  intake gasket leak/damage            10   
4             11             0  intake gasket leak/damage             0   
5             20             0  intake gasket leak/damage             0   
6             22             0  intake gasket leak/damage             0   
7             25             0  intake gasket leak/damage             0   
8             28             1  intake gasket leak/damage            10   
9             32       

In [6]:
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 加载数据
# ==========================================
data_dir = '/root'
data = pd.read_pickle(os.path.join(data_dir, 'flight_data.pkl'))
header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))
stats_df = pd.read_csv(os.path.join(data_dir, 'stats.csv'), index_col=0)

print(f"✅ 数据加载成功: data={len(data)} 航班, header={len(header_df)} 航班")

# ==========================================
# 2. 按论文 Section 3.2 标准筛选数据
# ==========================================
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())
filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"筛选后航班数: {len(filtered_header)}")

# ==========================================
# 3. 准备特征和标签
# ==========================================
target_len = 3000
X_list = []
y = []
flight_groups = []  # 记录每个航班所属的维护事件组

print("准备数据...")
for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    # 使用 stats.csv 全局归一化
    for i in range(sensor_data.shape[1]):
        col_name = f'sensor_{i}'
        if col_name in stats_df.index:
            min_val = stats_df.loc[col_name, 'min']
            max_val = stats_df.loc[col_name, 'max']
            if max_val > min_val:
                sensor_data[:, i] = (sensor_data[:, i] - min_val) / (max_val - min_val)
    
    current_len = sensor_data.shape[0]
    if current_len >= target_len:
        sensor_data = sensor_data[:target_len, :]
    else:
        pad_width = ((0, target_len - current_len), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)
    
    # 使用 date_diff 和 class 组合作为分组标识（同一维护事件的航班共享）
    group_key = f"{filtered_header.iloc[idx]['class']}_{abs(filtered_header.iloc[idx]['date_diff'])}"
    flight_groups.append(group_key)

X = np.array(X_list)
y = np.array(y)
flight_groups = np.array(flight_groups)

print(f"X 形状: {X.shape}")
print(f"标签分布: 0={sum(y==0)}, 1={sum(y==1)}")
print(f"唯一分组数: {len(np.unique(flight_groups))}")

# ==========================================
# 4. 基于分组的交叉验证
# ==========================================
from sklearn.model_selection import GroupKFold

# 使用 GroupKFold 确保同一维护事件的航班在同一折中
gkf = GroupKFold(n_splits=5)

minirocket = MiniRocketMultivariate(random_state=42)
classifier = LogisticRegressionCV(Cs=10, cv=3, random_state=42, max_iter=2000)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*50)
print("GroupKFold 交叉验证（按维护事件分组）")
print("="*50)

fold = 1
for train_idx, val_idx in gkf.split(X, y, groups=flight_groups):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # 检查是否包含两个类别
    if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2:
        print(f"⚠️ Fold {fold} 跳过（训练集或验证集只有一个类别）")
        continue
    
    print(f"\n--- Fold {fold} ---")
    print(f"训练集: {len(train_idx)} 样本, 验证集: {len(val_idx)} 样本")
    print(f"训练集标签: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
    print(f"验证集标签: 0={sum(y_val==0)}, 1={sum(y_val==1)}")
    
    start = time.time()
    
    X_train_transform = minirocket.fit_transform(X_train, y_train)
    classifier.fit(X_train_transform, y_train)
    X_val_transform = minirocket.transform(X_val)
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    
    elapsed = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

if len(accuracies) > 0:
    print("\n" + "="*50)
    print("GroupKFold 交叉验证最终结果:")
    print(f"准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
    print(f"F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
    print(f"AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
    print("="*50)
else:
    print("❌ 没有成功完成任何折的验证")

✅ 数据加载成功: data=11446 航班, header=11446 航班
筛选后航班数: 11446
准备数据...
X 形状: (11446, 3000, 23)
标签分布: 0=5844, 1=5602
唯一分组数: 38

GroupKFold 交叉验证（按维护事件分组）

--- Fold 1 ---
训练集: 8509 样本, 验证集: 2937 样本
训练集标签: 0=4182, 1=4327
验证集标签: 0=1662, 1=1275
准确率: 0.5346, F1: 0.5249, AUC: 0.5634, 耗时: 139.11s

--- Fold 2 ---
训练集: 9296 样本, 验证集: 2150 样本
训练集标签: 0=4636, 1=4660
验证集标签: 0=1208, 1=942
准确率: 0.5474, F1: 0.4956, AUC: 0.5604, 耗时: 174.61s

--- Fold 3 ---
训练集: 9327 样本, 验证集: 2119 样本
训练集标签: 0=4972, 1=4355
验证集标签: 0=872, 1=1247
准确率: 0.5352, F1: 0.5141, AUC: 0.5931, 耗时: 164.10s

--- Fold 4 ---
训练集: 9327 样本, 验证集: 2119 样本
训练集标签: 0=4842, 1=4485
验证集标签: 0=1002, 1=1117
准确率: 0.5507, F1: 0.5245, AUC: 0.5848, 耗时: 162.48s

--- Fold 5 ---
训练集: 9325 样本, 验证集: 2121 样本
训练集标签: 0=4744, 1=4581
验证集标签: 0=1100, 1=1021
准确率: 0.5427, F1: 0.5222, AUC: 0.5756, 耗时: 162.59s

GroupKFold 交叉验证最终结果:
准确率: 0.5421 ± 0.0065
F1分数: 0.5162 ± 0.0110
AUC: 0.5755 ± 0.0124


In [8]:
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sklearn.linear_model import LogisticRegressionCV

# ==========================================
# 1. 加载数据
# ==========================================
data_dir = '/root'
data = pd.read_pickle(os.path.join(data_dir, 'flight_data.pkl'))
header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))
stats_df = pd.read_csv(os.path.join(data_dir, 'stats.csv'), index_col=0)

print(f"✅ 数据加载成功")

# ==========================================
# 2. 筛选数据
# ==========================================
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())

# 只保留前两大类别（简化实验）
main_classes = ['intake gasket leak/damage', 'rocker cover leak/loose/damage']
mask = mask & (header_df['label'].isin(main_classes))

filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"筛选后航班数: {len(filtered_header)}")

# ==========================================
# 3. 准备特征和标签
# ==========================================
target_len = 4096
X_list = []
y = []

print("准备数据...")
for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    # 归一化
    for i in range(sensor_data.shape[1]):
        col_name = f'sensor_{i}'
        if col_name in stats_df.index:
            min_val = stats_df.loc[col_name, 'min']
            max_val = stats_df.loc[col_name, 'max']
            if max_val > min_val:
                sensor_data[:, i] = (sensor_data[:, i] - min_val) / (max_val - min_val)
    
    # 只截取最后 target_len 步
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list)
y = np.array(y)

print(f"X 形状: {X.shape}")
print(f"标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

# ==========================================
# 4. 使用 StratifiedKFold（简单可靠）
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

minirocket = MiniRocketMultivariate(random_state=42)
classifier = LogisticRegressionCV(Cs=10, cv=3, random_state=42, max_iter=5000)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*50)
print("StratifiedKFold 优化版 (target_len=4096, 只保留主要类别)")
print("="*50)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"训练集: {len(train_idx)} 样本, 验证集: {len(val_idx)} 样本")
    print(f"训练集标签: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
    print(f"验证集标签: 0={sum(y_val==0)}, 1={sum(y_val==1)}")
    
    start = time.time()
    
    X_train_transform = minirocket.fit_transform(X_train, y_train)
    classifier.fit(X_train_transform, y_train)
    X_val_transform = minirocket.transform(X_val)
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    
    elapsed = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*50)
print("优化版最终结果:")
print(f"准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*50)

✅ 数据加载成功
筛选后航班数: 6570
准备数据...
X 形状: (6570, 4096, 23)
标签分布: 0=3448, 1=3122

StratifiedKFold 优化版 (target_len=4096, 只保留主要类别)

--- Fold 1 ---
训练集: 5256 样本, 验证集: 1314 样本
训练集标签: 0=2759, 1=2497
验证集标签: 0=689, 1=625
准确率: 0.5320, F1: 0.4730, AUC: 0.5541, 耗时: 75.50s

--- Fold 2 ---
训练集: 5256 样本, 验证集: 1314 样本
训练集标签: 0=2759, 1=2497
验证集标签: 0=689, 1=625
准确率: 0.5487, F1: 0.5135, AUC: 0.5729, 耗时: 94.55s

--- Fold 3 ---
训练集: 5256 样本, 验证集: 1314 样本
训练集标签: 0=2758, 1=2498
验证集标签: 0=690, 1=624
准确率: 0.5457, F1: 0.5243, AUC: 0.5721, 耗时: 79.55s

--- Fold 4 ---
训练集: 5256 样本, 验证集: 1314 样本
训练集标签: 0=2758, 1=2498
验证集标签: 0=690, 1=624
准确率: 0.5731, F1: 0.5250, AUC: 0.5903, 耗时: 82.68s

--- Fold 5 ---
训练集: 5256 样本, 验证集: 1314 样本
训练集标签: 0=2758, 1=2498
验证集标签: 0=690, 1=624
准确率: 0.5723, F1: 0.5164, AUC: 0.5939, 耗时: 70.69s

优化版最终结果:
准确率: 0.5543 ± 0.0160
F1分数: 0.5104 ± 0.0192
AUC: 0.5766 ± 0.0143


In [18]:
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegressionCV
from sktime.transformations.panel.rocket import MiniRocketMultivariate

# ==========================================
# 1. 加载数据
# ==========================================
data_dir = '/root'
data = pd.read_pickle(os.path.join(data_dir, 'flight_data.pkl'))
header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))
stats_df = pd.read_csv(os.path.join(data_dir, 'stats.csv'), index_col=0)

print(f"✅ 数据加载成功")

# ==========================================
# 2. 筛选数据
# ==========================================
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())
filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"筛选后航班数: {len(filtered_header)}")

# ==========================================
# 3. 准备特征和标签
# ==========================================
target_len = 4096
X_list = []
y = []

print("准备数据...")
for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    # 归一化
    for i in range(sensor_data.shape[1]):
        col_name = f'sensor_{i}'
        if col_name in stats_df.index:
            min_val = stats_df.loc[col_name, 'min']
            max_val = stats_df.loc[col_name, 'max']
            if max_val > min_val:
                sensor_data[:, i] = (sensor_data[:, i] - min_val) / (max_val - min_val)
    
    # 截取最后 target_len 步
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y)

print(f"X 形状: {X.shape}")
print(f"标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

# ==========================================
# 4. 使用 sktime 的 MiniRocketMultivariate
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 修正参数名：n_kernels -> num_kernels
# 同时去掉 max_dilations_per_kernel（可能也不存在）
minirocket = MiniRocketMultivariate(
    random_state=42,
    num_kernels=10000,  # 修正：n_kernels -> num_kernels
)

classifier = LogisticRegressionCV(Cs=10, cv=3, random_state=42, max_iter=5000)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*50)
print("sktime MiniRocketMultivariate (调优) + 完整数据集")
print("="*50)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"训练集: {len(train_idx)} 样本, 验证集: {len(val_idx)} 样本")
    print(f"训练集标签: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
    print(f"验证集标签: 0={sum(y_val==0)}, 1={sum(y_val==1)}")
    
    start = time.time()
    
    X_train_transform = minirocket.fit_transform(X_train, y_train)
    classifier.fit(X_train_transform, y_train)
    X_val_transform = minirocket.transform(X_val)
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    
    elapsed = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*50)
print("sktime MiniRocketMultivariate 最终结果:")
print(f"准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*50)

✅ 数据加载成功
筛选后航班数: 11446
准备数据...
X 形状: (11446, 4096, 23)
标签分布: 0=5844, 1=5602

sktime MiniRocketMultivariate (调优) + 完整数据集

--- Fold 1 ---
训练集: 9156 样本, 验证集: 2290 样本
训练集标签: 0=4675, 1=4481
验证集标签: 0=1169, 1=1121
准确率: 0.5323, F1: 0.5178, AUC: 0.5529, 耗时: 187.69s

--- Fold 2 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4676, 1=4481
验证集标签: 0=1168, 1=1121
准确率: 0.5430, F1: 0.5293, AUC: 0.5654, 耗时: 227.31s

--- Fold 3 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4675, 1=4482
验证集标签: 0=1169, 1=1120
准确率: 0.5260, F1: 0.4951, AUC: 0.5494, 耗时: 182.00s

--- Fold 4 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4675, 1=4482
验证集标签: 0=1169, 1=1120
准确率: 0.5487, F1: 0.5264, AUC: 0.5753, 耗时: 170.43s

--- Fold 5 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4675, 1=4482
验证集标签: 0=1169, 1=1120
准确率: 0.5391, F1: 0.4983, AUC: 0.5490, 耗时: 173.69s

sktime MiniRocketMultivariate 最终结果:
准确率: 0.5378 ± 0.0080
F1分数: 0.5134 ± 0.0141
AUC: 0.5584 ± 0.0103


In [19]:
import numpy as np
import pandas as pd
import time
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 定义 ConvMHSA 模型
# ==========================================
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        attn_output, _ = self.attention(x, x, x)
        attn_output = self.dropout(attn_output)
        return self.norm(x + attn_output)

class ConvMHSA(nn.Module):
    def __init__(self, input_channels=23, seq_len=4096, embed_dim=64, num_heads=8, num_layers=4, num_classes=2):
        super().__init__()
        
        # 1D 卷积压缩时间维度: 4096 -> 512
        self.conv_layers = nn.Sequential(
            nn.Conv1d(input_channels, embed_dim, kernel_size=8, stride=4, padding=2),
            nn.ReLU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=8, stride=2, padding=3),
            nn.ReLU(),
        )
        # 计算压缩后的序列长度：4096 -> 512
        self.compressed_len = seq_len // 8  # 约 512
        
        # 位置编码
        self.pos_embedding = nn.Parameter(torch.randn(1, self.compressed_len, embed_dim) * 0.02)
        
        # Multi-Head Self-Attention 层
        self.attn_layers = nn.ModuleList([
            MultiHeadSelfAttention(embed_dim, num_heads) for _ in range(num_layers)
        ])
        
        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        # x: (batch, channels, length) -> (batch, embed_dim, compressed_len)
        x = self.conv_layers(x)
        # (batch, embed_dim, compressed_len) -> (batch, compressed_len, embed_dim)
        x = x.permute(0, 2, 1)
        
        # 添加位置编码
        x = x + self.pos_embedding
        
        # Self-Attention
        for attn in self.attn_layers:
            x = attn(x)
        
        # 全局平均池化
        x = x.mean(dim=1)
        
        # 分类
        return self.classifier(x)

# ==========================================
# 2. 训练函数
# ==========================================
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(y_batch.numpy())
    
    return np.array(all_preds), np.array(all_probs), np.array(all_labels)

# ==========================================
# 3. 加载数据
# ==========================================
data_dir = '/root'
data = pd.read_pickle(os.path.join(data_dir, 'flight_data.pkl'))
header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))
stats_df = pd.read_csv(os.path.join(data_dir, 'stats.csv'), index_col=0)

print(f"✅ 数据加载成功")

# ==========================================
# 4. 筛选数据
# ==========================================
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())
filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"筛选后航班数: {len(filtered_header)}")

# ==========================================
# 5. 准备特征和标签
# ==========================================
target_len = 4096
X_list = []
y = []

print("准备数据...")
for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    # 归一化
    for i in range(sensor_data.shape[1]):
        col_name = f'sensor_{i}'
        if col_name in stats_df.index:
            min_val = stats_df.loc[col_name, 'min']
            max_val = stats_df.loc[col_name, 'max']
            if max_val > min_val:
                sensor_data[:, i] = (sensor_data[:, i] - min_val) / (max_val - min_val)
    
    # 截取最后 target_len 步
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y)

print(f"X 形状: {X.shape}")
print(f"标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

# ==========================================
# 6. ConvMHSA 交叉验证
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*50)
print("ConvMHSA 交叉验证 (target_len=4096)")
print("="*50)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"训练集: {len(train_idx)} 样本, 验证集: {len(val_idx)} 样本")
    
    # 转换为 PyTorch Tensor
    # ConvMHSA 期望输入: (batch, channels, length)
    X_train_t = torch.tensor(np.transpose(X_train, (0, 2, 1)), dtype=torch.float32)
    X_val_t = torch.tensor(np.transpose(X_val, (0, 2, 1)), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    
    # 创建 DataLoader
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    # 初始化模型
    model = ConvMHSA(
        input_channels=23,
        seq_len=target_len,
        embed_dim=64,
        num_heads=8,
        num_layers=4,
        num_classes=2
    ).to(device)
    
    # 优化器和损失函数
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
    criterion = nn.CrossEntropyLoss()
    
    # 训练
    start = time.time()
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    
    for epoch in range(50):  # 50 epochs
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # 验证
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    elapsed = time.time() - start
    
    # 评估
    y_pred, y_prob, _ = evaluate(model, val_loader, device)
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*50)
print("ConvMHSA 最终结果:")
print(f"准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*50)

✅ 数据加载成功
筛选后航班数: 11446
准备数据...
X 形状: (11446, 4096, 23)
标签分布: 0=5844, 1=5602
使用设备: cuda

ConvMHSA 交叉验证 (target_len=4096)

--- Fold 1 ---
训练集: 9156 样本, 验证集: 2290 样本
准确率: 0.5140, F1: 0.5150, AUC: 0.5180, 耗时: 111.45s

--- Fold 2 ---
训练集: 9157 样本, 验证集: 2289 样本
准确率: 0.5256, F1: 0.2119, AUC: 0.5214, 耗时: 190.47s

--- Fold 3 ---
训练集: 9157 样本, 验证集: 2289 样本
准确率: 0.5203, F1: 0.1891, AUC: 0.5191, 耗时: 212.66s

--- Fold 4 ---
训练集: 9157 样本, 验证集: 2289 样本
准确率: 0.4915, F1: 0.5705, AUC: 0.5034, 耗时: 190.54s

--- Fold 5 ---
训练集: 9157 样本, 验证集: 2289 样本
准确率: 0.5208, F1: 0.3323, AUC: 0.5256, 耗时: 233.02s

ConvMHSA 最终结果:
准确率: 0.5144 ± 0.0120
F1分数: 0.3638 ± 0.1550
AUC: 0.5175 ± 0.0075


In [21]:
import numpy as np
import pandas as pd
import time
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 导入 tsai 的 InceptionTime
from tsai.models.InceptionTime import InceptionTime

# ==========================================
# 1. 加载数据
# ==========================================
data_dir = '/root'
data = pd.read_pickle(os.path.join(data_dir, 'flight_data.pkl'))
header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))
stats_df = pd.read_csv(os.path.join(data_dir, 'stats.csv'), index_col=0)

print(f"✅ 数据加载成功")

# ==========================================
# 2. 筛选数据
# ==========================================
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())
filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"筛选后航班数: {len(filtered_header)}")

# ==========================================
# 3. 准备特征和标签
# ==========================================
target_len = 4096
X_list = []
y = []

print("准备数据...")
for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    # 归一化
    for i in range(sensor_data.shape[1]):
        col_name = f'sensor_{i}'
        if col_name in stats_df.index:
            min_val = stats_df.loc[col_name, 'min']
            max_val = stats_df.loc[col_name, 'max']
            if max_val > min_val:
                sensor_data[:, i] = (sensor_data[:, i] - min_val) / (max_val - min_val)
    
    # 截取最后 target_len 步
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y)

print(f"X 形状: {X.shape}")
print(f"标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

# ==========================================
# 4. 训练函数
# ==========================================
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(y_batch.numpy())
    
    return np.array(all_preds), np.array(all_probs), np.array(all_labels)

# ==========================================
# 5. 使用 tsai 的 InceptionTime（手动训练）
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*50)
print("tsai InceptionTime 交叉验证 (target_len=4096)")
print("="*50)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"训练集: {len(train_idx)} 样本, 验证集: {len(val_idx)} 样本")
    print(f"训练集标签: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
    print(f"验证集标签: 0={sum(y_val==0)}, 1={sum(y_val==1)}")
    
    start = time.time()
    
    # InceptionTime 期望输入: (batch, channels, length)
    X_train_t = torch.tensor(np.transpose(X_train, (0, 2, 1)), dtype=torch.float32)
    X_val_t = torch.tensor(np.transpose(X_val, (0, 2, 1)), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    
    # 创建 DataLoader
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    # 创建模型
    model = InceptionTime(
        c_in=23,           # 输入通道数
        c_out=2,           # 输出类别数
        seq_len=target_len # 序列长度
    ).to(device)
    
    # 优化器和损失函数
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    # 训练
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0
    
    for epoch in range(30):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # 验证
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    elapsed = time.time() - start
    
    # 评估
    y_pred, y_prob, _ = evaluate(model, val_loader, device)
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*50)
print("tsai InceptionTime 最终结果:")
print(f"准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*50)

✅ 数据加载成功
筛选后航班数: 11446
准备数据...
X 形状: (11446, 4096, 23)
标签分布: 0=5844, 1=5602
使用设备: cuda

tsai InceptionTime 交叉验证 (target_len=4096)

--- Fold 1 ---
训练集: 9156 样本, 验证集: 2290 样本
训练集标签: 0=4675, 1=4481
验证集标签: 0=1169, 1=1121
准确率: 0.5498, F1: 0.2323, AUC: 0.6286, 耗时: 108.16s

--- Fold 2 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4676, 1=4481
验证集标签: 0=1168, 1=1121
准确率: 0.5299, F1: 0.6650, AUC: 0.6033, 耗时: 120.41s

--- Fold 3 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4675, 1=4482
验证集标签: 0=1169, 1=1120
准确率: 0.5129, F1: 0.0685, AUC: 0.5380, 耗时: 72.25s

--- Fold 4 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4675, 1=4482
验证集标签: 0=1169, 1=1120
准确率: 0.5544, F1: 0.6631, AUC: 0.6469, 耗时: 180.60s

--- Fold 5 ---
训练集: 9157 样本, 验证集: 2289 样本
训练集标签: 0=4675, 1=4482
验证集标签: 0=1169, 1=1120
准确率: 0.5212, F1: 0.2399, AUC: 0.5587, 耗时: 71.44s

tsai InceptionTime 最终结果:
准确率: 0.5336 ± 0.0161
F1分数: 0.3738 ± 0.2448
AUC: 0.5951 ± 0.0411
